<a href="https://colab.research.google.com/github/landpd/Generador-Materiales-Pulppo/blob/main/Descarga_de_fotograf%C3%ADas_1_5_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELDA 1: DESCARGA Y PROCESAMIENTO DE FOTOS
# ==========================================

import os
import glob
import re
import io
import requests
import pandas as pd
from PIL import Image
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir las rutas (Ajusta mayúsculas/minúsculas si es necesario)
RUTA_CSV = "/content/drive/MyDrive/Nuevos proyectos/CSV Metabase"
RUTA_FOTOS = "/content/drive/MyDrive/Nuevos proyectos/Fotografías Propiedades 1-5-10"

# 3. Función para procesar y guardar la imagen
def process_and_save_image(url, save_path):
    try:
        # Descargar la imagen
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        # Abrir imagen con Pillow
        img = Image.open(io.BytesIO(response.content))

        # Convertir a RGB (elimina transparencias si es PNG y evita errores al guardar como JPG)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")

        # Redimensionar si el lado más corto es mayor a 2000 pixeles
        w, h = img.size
        shortest_side = min(w, h)

        if shortest_side > 2000:
            ratio = 2000 / shortest_side
            new_w = int(w * ratio)
            new_h = int(h * ratio)
            img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)

        # Comprimir hasta que pese menos de 500 KB
        quality = 95
        while quality >= 10:
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=quality)
            size_kb = len(buffer.getvalue()) / 1024

            if size_kb <= 500:
                with open(save_path, "wb") as f:
                    f.write(buffer.getvalue())
                return True
            quality -= 5

        # Si aún bajando la calidad a 10 no se logra (raro), se guarda con calidad 10
        with open(save_path, "wb") as f:
            f.write(buffer.getvalue())
        return True

    except Exception as e:
        print(f"     [!] Error procesando {url}: {e}")
        return False

# 4. Encontrar el archivo CSV más reciente en la RUTA_CSV
csv_files = glob.glob(os.path.join(RUTA_CSV, "propiedades_1_5_10_con_fotos_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No se encontró ningún archivo CSV en: {RUTA_CSV}")

latest_csv = sorted(csv_files)[-1]
print(f"Archivo CSV detectado: {os.path.basename(latest_csv)}\n")

# 5. Leer CSV y procesar filas
df = pd.read_csv(latest_csv)

for index, row in df.iterrows():
    internal_id = str(row['InternalId']).strip()
    company_name = str(row['Company: Name']).strip()
    pictures_raw = str(row['Pictures'])

    # Ignorar si no hay datos de fotos o empresa
    if pd.isna(row['Pictures']) or pictures_raw.lower() == 'nan':
        continue

    # Extraer URLs usando Regex
    urls = re.findall(r'\[:url\s+"([^"]+)"\]', pictures_raw)
    total_fotos = len(urls)

    if total_fotos == 0:
        continue

    print(f"Procesando {internal_id} ({company_name}) - {total_fotos} fotos encontradas...")

    # Crear carpeta de la propiedad en la RUTA_FOTOS: Company / InternalId
    property_folder = os.path.join(RUTA_FOTOS, company_name, internal_id)
    os.makedirs(property_folder, exist_ok=True)

    # Procesar cada foto
    for i, url in enumerate(urls):
        # Generar el nombre de archivo (ej. ABC-123-photo-01-de-05.jpg)
        filename = f"{internal_id}-photo-{i+1:02d}-de-{total_fotos:02d}.jpg"
        save_path = os.path.join(property_folder, filename)

        # Saltar si la foto ya existe (para reanudar sin duplicar trabajo)
        if not os.path.exists(save_path):
            process_and_save_image(url, save_path)

print("\n¡Proceso de descarga y optimización completado!")

In [ ]:
# ==========================================
# CELDA 2: DESCARGA DE LOGOTIPOS DE INMOBILIARIAS
# ==========================================

import os
import glob
import re
import io
import requests
import pandas as pd
from PIL import Image
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir las rutas
RUTA_CSV = "/content/drive/MyDrive/Nuevos proyectos"
# Carpeta donde se guardarán los logos descargados (puedes cambiar el nombre si lo deseas)
RUTA_LOGOS = "/content/drive/MyDrive/Nuevos proyectos/Logos Inmobiliarias"

# Crear la carpeta destino si no existe
os.makedirs(RUTA_LOGOS, exist_ok=True)

# 3. Función para descargar y guardar el logo en PNG
def download_and_save_logo(url, save_path):
    try:
        # Descargar la imagen
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        # Abrir imagen con Pillow para asegurar que el formato sea procesado correctamente
        img = Image.open(io.BytesIO(response.content))

        # Guardar forzosamente como PNG (conservando transparencias originales)
        img.save(save_path, format="PNG")
        return True

    except Exception as e:
        print(f"     [!] Error procesando {url}: {e}")
        return False

# 4. Encontrar el archivo CSV más reciente en la RUTA_CSV
csv_files = glob.glob(os.path.join(RUTA_CSV, "logos___inmobiliarias_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No se encontró ningún archivo CSV con el patrón 'logos___inmobiliarias_*.csv' en: {RUTA_CSV}")

latest_csv = sorted(csv_files)[-1]
print(f"Archivo CSV detectado: {os.path.basename(latest_csv)}\n")

# 5. Leer CSV y procesar filas
df = pd.read_csv(latest_csv)

for index, row in df.iterrows():
    # Usamos iloc[0] y iloc[1] para garantizar que toma la 1ra (name) y 2da columna (enlace web)
    # sin importar si el encabezado de la URL cambia de nombre.
    raw_name = str(row.iloc[0]).strip()
    url = str(row.iloc[1]).strip()

    # Ignorar si no hay datos de nombre o URL validos
    if pd.isna(row.iloc[0]) or pd.isna(row.iloc[1]) or raw_name.lower() == 'nan' or url.lower() == 'nan':
        continue

    # Limpiar el nombre para quitar caracteres no válidos en archivos (ej. barras, comillas)
    safe_name = re.sub(r'[\\/*?:"<>|]', "", raw_name)

    # Generar el nombre de archivo (Estructura: Name + imagotipo_colab_negro.png)
    # Nota: Si necesitas un espacio entre el nombre y 'imagotipo', cambia la línea de abajo a:
    # filename = f"{safe_name} imagotipo_colab_negro.png"
    filename = f"{safe_name}_imagotipo_colab_negro.png"
    save_path = os.path.join(RUTA_LOGOS, filename)

    # Saltar si el logo ya existe (útil por si el proceso se interrumpe y lo reanudas)
    if not os.path.exists(save_path):
        print(f"Descargando logo de: {raw_name}...")
        download_and_save_logo(url, save_path)
    else:
        print(f"    -> El logo de {raw_name} ya existe. Omitiendo...")

print("\n¡Proceso de descarga de logotipos completado!")

In [ ]:
# ==========================================
# CELDA C: GENERADOR DE CSV PARA AFTER EFFECTS
# ==========================================

import os
import glob
import re
import pandas as pd
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir rutas
RUTA_CSV = "/content/drive/MyDrive/Nuevos proyectos/CSV Metabase"
RUTA_GUARDADO = "/content/drive/MyDrive/Nuevos proyectos/CSV_After_Effects.csv"

# 3. Buscar el CSV de Pulppo más reciente
csv_files = glob.glob(os.path.join(RUTA_CSV, "propiedades_1_5_10_con_fotos_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No se encontró ningún archivo CSV en: {RUTA_CSV}")

latest_csv = sorted(csv_files)[-1]
print(f"📄 Procesando base de datos: {os.path.basename(latest_csv)}\n")

df = pd.read_csv(latest_csv)

# --- Funciones de ayuda para formatear plurales y vacíos ---
def formatear_plural(valor, singular, plural):
    val_str = str(valor).strip()
    # Si está vacío o es 0, devolvemos texto vacío (Para que AE oculte los guiones)
    if val_str.lower() in ['nan', 'none', '', '0', '0.0']:
        return ""
    try:
        v_float = float(val_str)
        if v_float == 0:
            return ""
        # Si es un número entero (ej. 3.0), le quitamos el decimal
        if v_float.is_integer():
            v_int = int(v_float)
            return f"{v_int} {singular}" if v_int == 1 else f"{v_int} {plural}"
        else:
            return f"{v_float} {singular}" if v_float == 1 else f"{v_float} {plural}"
    except:
        return ""

def formatear_metros(valor):
    val_str = str(valor).strip()
    if val_str.lower() in ['nan', 'none', '', '0', '0.0']:
        return ""
    try:
        v_float = float(val_str)
        if v_float == 0:
            return ""
        if v_float.is_integer():
            return f"{int(v_float)} m² TOTALES"
        else:
            return f"{v_float} m² TOTALES"
    except:
        return ""

# 4. Preparar la lista de datos
datos_ae = []

for index, row in df.iterrows():
    internal_id = str(row['InternalId']).strip()
    if internal_id == 'nan': continue

    # --- NOMBRE DEL VIDEO (Composición en AE) ---
    company_raw = str(row.get('Company: Name', 'Inmobiliaria')).strip()
    if company_raw.lower() == 'nan': company_raw = "Inmobiliaria"
    company_clean = re.sub(r'[^A-Za-z0-9]+', '_', company_raw).strip('_')
    nombre_comp = f"{internal_id}_{company_clean}_(4-5)_Post"

    # --- CALLE Y COLONIA/CIUDAD ---
    # Calle
    calle_cruda = str(row.get('Address: Street', '')).strip()
    calle = calle_cruda if calle_cruda.lower() != 'nan' else ""

    # Colonia, Ciudad (CORREGIDO)
    colonia = str(row.get('Address: Neighborhood: Name', '')).strip()
    if colonia.lower() == 'nan': colonia = ""

    ciudad = str(row.get('Address: City: Name', '')).strip()
    if ciudad.lower() == 'nan': ciudad = ""

    # Unimos con coma, el strip evita que quede ", Ciudad" o "Colonia," si falta uno
    colonia_estado = f"{colonia}, {ciudad}".strip(", ")
    # En caso de que haya quedado una coma con espacio en los extremos
    if colonia_estado.startswith(", "): colonia_estado = colonia_estado[2:]
    if colonia_estado.endswith(","): colonia_estado = colonia_estado[:-1]

    # --- OPERACIÓN CON CORCHETES ---
    tipo = str(row.get('Type', '')).strip()
    if tipo.lower() == 'nan': tipo = ""

    operacion = str(row.get('Listing: Operation', '')).strip()
    if operacion.lower() == 'nan': operacion = ""
    if operacion.lower() == 'sale': operacion = 'Venta'
    elif operacion.lower() == 'rent': operacion = 'Renta'

    if tipo and operacion:
        tipo_en_operacion = f"[ {tipo} en {operacion} ]"
    elif tipo:
        tipo_en_operacion = f"[ {tipo} ]"
    elif operacion:
        tipo_en_operacion = f"[ {operacion} ]"
    else:
        tipo_en_operacion = ""

    # --- PRECIO (Mantenido por si decides agregarlo al video después) ---
    precio_crudo = str(row.get('Listing: Price: Price', '0')).strip()
    moneda_crudo = str(row.get('Listing: Price: Currency', 'mxn')).strip().lower()
    if moneda_crudo == 'nan' or moneda_crudo == '': moneda_crudo = "mxn"
    precio_limpio = precio_crudo.replace(',', '').replace('$', '').replace(' ', '')
    try:
        precio_formateado = f"${float(precio_limpio):,.2f} {moneda_crudo}"
    except:
        precio_formateado = precio_crudo if precio_crudo.lower() != 'nan' else ""

    # --- ATRIBUTOS FORMATEADOS ---
    # Automáticamente se encargará de singular/plural o devolver "" si es 0
    habitaciones = formatear_plural(row.get('Attributes: Suites', ''), "HABITACIÓN", "HABITACIONES")
    banos = formatear_plural(row.get('Attributes: Bathrooms', ''), "BAÑO", "BAÑOS")
    estacionamientos = formatear_plural(row.get('Attributes: Parkings', ''), "ESTACIONAMIENTO", "ESTACIONAMIENTOS")
    metros = formatear_metros(row.get('Attributes: TotalSurface', ''))

    # --- RECONSTRUIR NOMBRES DE ARCHIVOS MULTIMEDIA ---
    pictures_raw = str(row['Pictures'])
    urls = re.findall(r'\[:url\s+"([^"]+)"\]', pictures_raw)
    total_fotos = len(urls)

    media_files = {}
    for i in range(10): # Rango para Media_01 a Media_10
        num_media = i + 1
        key = f"Media_{num_media:02d}"

        if i < total_fotos:
            filename = f"{internal_id}-photo-{num_media:02d}-de-{total_fotos:02d}.jpg"
            media_files[key] = filename
        else:
            media_files[key] = ""

    # --- ENSAMBLAR LA FILA ---
    fila = {
        "Name": nombre_comp,
        "Calle": calle,
        "Colonia, Estado": colonia_estado,
        "inmueble en operacion": tipo_en_operacion,
        "Precio": precio_formateado,
        "habitaciones": habitaciones,
        "baños": banos,
        "estacionamientos": estacionamientos,
        "metros totales": metros
    }

    fila.update(media_files)
    datos_ae.append(fila)

# 5. Generar el CSV final
df_ae = pd.DataFrame(datos_ae)

# Codificación utf-8-sig crucial para AE y los acentos
df_ae.to_csv(RUTA_GUARDADO, index=False, encoding='utf-8-sig')

print(f"✅ ¡CSV para After Effects generado con éxito!")
print(f"📂 Archivo guardado en: {RUTA_GUARDADO}")

display(df_ae[['Name', 'Calle', 'Colonia, Estado', 'inmueble en operacion', 'habitaciones', 'baños']].head())

In [ ]:
# ==========================================
# CELDA D: ORGANIZADOR DE DISEÑOS GENERADOS
# ==========================================

import os
import glob
import shutil
import pandas as pd
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Rutas
RUTA_CSV = "/content/drive/MyDrive/Nuevos proyectos/CSV Metabase"
RUTA_DISENOS = "/content/drive/MyDrive/Nuevos proyectos/Diseños_Generados"

# 3. Buscar el CSV más reciente para mapear InternalId -> Company: Name
csv_files = glob.glob(os.path.join(RUTA_CSV, "propiedades_1_5_10_con_fotos_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No se encontró ningún archivo CSV en: {RUTA_CSV}")

latest_csv = sorted(csv_files)[-1]
print(f"📄 Leyendo base de datos para mapear Inmobiliarias: {os.path.basename(latest_csv)}")

df = pd.read_csv(latest_csv)

# Crear diccionario { "ABC-123": "Inmobiliaria Del Valle" }
mapeo_inmobiliarias = {}
for index, row in df.iterrows():
    internal_id = str(row['InternalId']).strip()
    if internal_id == 'nan': continue

    company_raw = str(row.get('Company: Name', 'Inmobiliaria')).strip()
    if company_raw.lower() == 'nan': company_raw = "Inmobiliaria"

    mapeo_inmobiliarias[internal_id] = company_raw

# 4. Organizar los archivos en las carpetas
print("\n🚀 Iniciando organización de archivos...\n")
archivos_movidos = 0

# Revisamos todo lo que hay en la carpeta de Diseños Generados
for archivo in os.listdir(RUTA_DISENOS):
    ruta_origen = os.path.join(RUTA_DISENOS, archivo)

    # Ignoramos si es una carpeta (por si ya corriste el script antes o creaste carpetas)
    if os.path.isdir(ruta_origen):
        continue

    # Extraemos el InternalId del nombre del archivo.
    # Como el archivo se llama "ABC-123_Empresa_post.jpg", cortamos por el primer guion bajo "_"
    partes = archivo.split('_')
    posible_id = partes[0]

    # Si el ID existe en nuestro mapeo, procedemos a moverlo
    if posible_id in mapeo_inmobiliarias:
        nombre_inmobiliaria = mapeo_inmobiliarias[posible_id]

        # Construimos la ruta de destino: Diseños_Generados / Company Name / InternalId
        carpeta_destino = os.path.join(RUTA_DISENOS, nombre_inmobiliaria, posible_id)

        # Creamos la carpeta si no existe
        os.makedirs(carpeta_destino, exist_ok=True)

        # Ruta final del archivo
        ruta_destino = os.path.join(carpeta_destino, archivo)

        # Movemos el archivo
        shutil.move(ruta_origen, ruta_destino)
        archivos_movidos += 1
        print(f"📦 Movido: {archivo} -> {nombre_inmobiliaria}/{posible_id}/")

print(f"\n✅ ¡Organización completada! Se ordenaron {archivos_movidos} diseños en sus respectivas carpetas.")